# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "MPPT_PREDICTION_PV_DATA_CSV": project_path("MPPT Prediction PV Data.csv"),
}

## Import and Inspect the Data

In [ ]:
import pandas as pd

data = pd.read_csv(INPUT_PATHS["MPPT_PREDICTION_PV_DATA_CSV"])
data.head(10)

## Preprocess Data


In [ ]:
data_frame = pd.DataFrame(data)
data_frame

In [ ]:
# Creating features and labels
X = data_frame.drop("Pmax", axis=1)
y = data_frame["Pmax"]
X.shape, y.shape

In [ ]:
# Creating training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)
len(X), len(X_train), len(X_test)

In [ ]:
X_train.head()

## Build first model

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Set random seed
tf.random.set_seed(42)

# Create model (model_1)
from experiment_mppt_prediction import build_model_1_cell_10

model_1 = build_model_1_cell_10()

# Compile the model
model_1.compile(
    loss="mae",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=["mae"],
)

# Fit the model
history_1 = model_1.fit(X_train, y_train, epochs=100)

## Evaluate the model

In [ ]:
# Plot training curve
pd.DataFrame(history_1.history).plot()
plt.xlabel("epochs")
plt.ylabel("loss")

In [ ]:
# Evaluate the model
model_1.evaluate(X_test, y_test)

# Build second model with normalized data


## Preprocess data

In [ ]:
data_frame

In [ ]:
(
    data_frame["V1"].max(),
    data_frame["P"].max(),
    data_frame["G"].max(),
    data_frame["Vopt"].max(),
)

In [ ]:
X["V1"] = X["V1"] / 45
X["P"] = X["P"] / 292.7
X["G"] = X["G"] / 1000
X["Vopt"] = X["Vopt"] / 38.9
X["V1"].max(), X["P"].max(), X["G"].max(), X["Vopt"].max()

In [ ]:
y.max()
y = y / 292.7
y.max()

In [ ]:
# Creating training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)
len(X), len(X_train), len(X_test)

## Build second model (model_2)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Set random seed
tf.random.set_seed(42)

# Create model (model_1)
from experiment_mppt_prediction import build_model_2_cell_22

model_2 = build_model_2_cell_22()

# Compile the model
model_2.compile(
    loss="mse",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=["mse"],
)

# Fit the model
history_2 = model_2.fit(X_train, y_train, epochs=50)

## Evaluate our model

In [ ]:
# Evaluate the model
model_2.evaluate(X_test, y_test)

In [ ]:
# Plot training curve
pd.DataFrame(history_2.history).plot()
plt.xlabel("epochs")
plt.ylabel("loss")

In [ ]:
y_preds = model_2.predict(X_test)

In [ ]:
y_preds

In [ ]:
y_test = tf.squeeze(y_test)

In [ ]:
y_test

In [ ]:
y_preds = tf.squeeze(y_preds)

In [ ]:
y_preds

In [ ]:
s1 = y_test * 292.7
s2 = y_preds * 292.7
s1, s2

In [ ]:
s1[100], s2[100]